# Advanced Tool Design — Building Robust Agent Tools

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/13_advanced_tool_design.ipynb)

## What This Notebook Teaches You

In Notebook 04 you built basic tools — simple functions the agent could call. But production tools need **much more**: input validation, error recovery, state management, composability, and testing. The difference between a demo tool and a production tool is like the difference between a script and a library.

**Research foundation**:
- Schick et al. 2023, *"Toolformer: Language Models Can Teach Themselves to Use Tools"* — demonstrated that LLMs can learn tool use from examples when tools are well-designed
- Qin et al. 2023, *"ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world APIs"* — showed tool schema design is critical for reliable tool selection
- Patil et al. 2023, *"Gorilla: Large Language Model Connected with Massive APIs"* — found that clear documentation in tool descriptions dramatically improves call accuracy

By the end of this notebook, you will:

1. **Understand tool design principles** — single responsibility, clear schemas, predictable errors
2. **Build a ToolDefinition dataclass** with full parameter and return type schemas
3. **Implement an advanced ToolRegistry** with validation, discovery, and error handling
4. **Create complex tool types** — stateful, confirmation-required, and composed tools
5. **Build robust input validation** with LLM-friendly error messages
6. **Test tools systematically** with a testing harness
7. **Wire 10+ production tools** into an agent loop
8. **Compare simple vs. production tools** on error recovery and reliability

---

> **Prerequisites**: Notebooks 01–04 (agent fundamentals, tool use).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~55–70 minutes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **Why Tool Design Matters**
- Understand **The ToolDefinition Dataclass**
- Understand **Advanced Tool Registry**


## Part 0: Environment Setup

Same Qwen3-8B with 4-bit quantization. The model will act as an agent that selects and uses our advanced tools.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Tool Design Matters

### 1.1 — The Problem with Naive Tools

Most tutorials show tools like this:

```python
def calculator(expression):
    return eval(expression)
```

This "works" but fails in production because:
- **No input validation** — what happens with `calculator("import os")`?
- **No error handling** — `calculator("1/0")` crashes the agent
- **No documentation** — the LLM has to guess what formats are accepted
- **No type safety** — anything goes in, anything comes out
- **No testing** — you discover bugs at runtime, in front of users

### 1.2 — What Production Tools Need

```
┌─────────────────────────────────────────┐
│            PRODUCTION TOOL              │
├─────────────────────────────────────────┤
│  📋 Schema     │ name, description,     │
│                │ parameters, returns    │
├────────────────┼────────────────────────┤
│  ✅ Validation │ type checks, ranges,   │
│                │ required fields        │
├────────────────┼────────────────────────┤
│  🛡️ Errors    │ structured, helpful,   │
│                │ recoverable            │
├────────────────┼────────────────────────┤
│  📖 Docs      │ examples in docstring, │
│                │ usage hints            │
├────────────────┼────────────────────────┤
│  🧪 Tests     │ happy path, edge,      │
│                │ error cases            │
└─────────────────────────────────────────┘
```

### 1.3 — Design Principles

1. **Single Responsibility** — one tool does one thing well
2. **Clear Schemas** — the LLM knows exactly what to pass and what to expect
3. **Predictable Errors** — structured error responses, never raw exceptions
4. **Usage Examples** — docstrings with examples are the best "few-shot prompts" for tool use
5. **Fail Gracefully** — return error info, don't crash the agent loop

## 2. The ToolDefinition Dataclass

Every tool gets a formal definition. Think of this like an OpenAPI spec — it tells the LLM everything it needs to call the tool correctly.

In [ ]:
@dataclass
class ParameterSchema:
    """Schema for a single tool parameter."""
    name: str
    type: str           # "string", "number", "integer", "boolean", "list", "dict"
    description: str
    required: bool = True
    default: Any = None
    enum: Optional[List[str]] = None     # allowed values
    min_value: Optional[float] = None    # for numbers
    max_value: Optional[float] = None    # for numbers
    pattern: Optional[str] = None        # regex for strings

    def to_dict(self) -> dict:
        d = {"name": self.name, "type": self.type, "description": self.description,
             "required": self.required}
        if self.default is not None: d["default"] = self.default
        if self.enum: d["enum"] = self.enum
        if self.min_value is not None: d["min_value"] = self.min_value
        if self.max_value is not None: d["max_value"] = self.max_value
        return d

@dataclass
class ToolDefinition:
    """Complete definition for a production tool."""
    name: str
    description: str
    parameters: List[ParameterSchema]
    return_type: str       # "string", "number", "dict", "list", etc.
    return_description: str
    examples: List[Dict[str, Any]] = field(default_factory=list)
    category: str = "general"
    version: str = "1.0"

    def to_schema_dict(self) -> dict:
        """Convert to JSON schema for LLM consumption."""
        return {
            "name": self.name,
            "description": self.description,
            "parameters": [p.to_dict() for p in self.parameters],
            "returns": {"type": self.return_type, "description": self.return_description},
            "examples": self.examples,
            "category": self.category,
        }

    def to_prompt_string(self) -> str:
        """Format for inclusion in system prompts."""
        lines = [f"### {self.name}", f"{self.description}\n", "Parameters:"]
        for p in self.parameters:
            req = "required" if p.required else f"optional, default={p.default}"
            lines.append(f"  - {p.name} ({p.type}, {req}): {p.description}")
            if p.enum:
                lines.append(f"    Allowed values: {p.enum}")
            if p.min_value is not None or p.max_value is not None:
                lines.append(f"    Range: [{p.min_value}, {p.max_value}]")
        lines.append(f"\nReturns ({self.return_type}): {self.return_description}")
        if self.examples:
            lines.append("\nExamples:")
            for ex in self.examples:
                lines.append(f'  Input:  {json.dumps(ex.get("input", {}))}')
                lines.append(f'  Output: {json.dumps(ex.get("output", ""))}')
        return "\n".join(lines)

# Demonstrate with a calculator tool definition
calc_def = ToolDefinition(
    name="calculator",
    description="Evaluate a mathematical expression safely. Supports +, -, *, /, **, sqrt, abs, and common math functions.",
    parameters=[
        ParameterSchema("expression", "string", "Mathematical expression to evaluate", required=True,
                        pattern=r"^[0-9+\-*/().\s a-z]+$"),
    ],
    return_type="dict",
    return_description="Dictionary with 'result' (number) and 'expression' (string echo)",
    examples=[
        {"input": {"expression": "2 + 3 * 4"}, "output": {"result": 14, "expression": "2 + 3 * 4"}},
        {"input": {"expression": "sqrt(144)"}, "output": {"result": 12.0, "expression": "sqrt(144)"}},
    ],
    category="math",
)

print("=== Tool Schema (JSON) ===")
print(json.dumps(calc_def.to_schema_dict(), indent=2)[:600])
print("\n=== Prompt Format ===")
print(calc_def.to_prompt_string())

## 3. Advanced Tool Registry

The registry is the central hub: it stores tools, validates them at registration time, dispatches calls, and handles errors. Think of it as a strongly-typed function dispatch table.

In [ ]:
class ToolResult:
    """Standardized result from any tool call."""
    def __init__(self, success: bool, result: Any = None, error: str = None,
                 tool_name: str = "", execution_time: float = 0.0):
        self.success = success
        self.result = result
        self.error = error
        self.tool_name = tool_name
        self.execution_time = execution_time

    def to_dict(self) -> dict:
        d = {"success": self.success, "tool": self.tool_name,
             "execution_time_ms": round(self.execution_time * 1000, 1)}
        if self.success:
            d["result"] = self.result
        else:
            d["error"] = self.error
        return d

    def __repr__(self):
        if self.success:
            return f"ToolResult(success=True, result={self.result!r})"
        return f"ToolResult(success=False, error={self.error!r})"


class ToolRegistry:
    """Advanced registry with validation, discovery, and error handling."""

    TYPE_MAP = {
        "string": str, "number": (int, float), "integer": int,
        "boolean": bool, "list": list, "dict": dict,
    }

    def __init__(self):
        self.tools: Dict[str, Callable] = {}
        self.definitions: Dict[str, ToolDefinition] = {}
        self.call_history: List[dict] = []

    def register(self, definition: ToolDefinition, func: Callable) -> None:
        """Register a tool with its definition. Validates the definition."""
        assert definition.name, "Tool must have a name"
        assert definition.description, "Tool must have a description"
        assert callable(func), "Tool function must be callable"
        if definition.name in self.tools:
            print(f"⚠️  Overwriting existing tool: {definition.name}")
        self.tools[definition.name] = func
        self.definitions[definition.name] = definition
        print(f"✓ Registered tool: {definition.name} (v{definition.version}, category={definition.category})")

    def validate_input(self, tool_name: str, kwargs: dict) -> Optional[str]:
        """Validate inputs against the tool's parameter schema. Returns error string or None."""
        defn = self.definitions.get(tool_name)
        if not defn:
            return f"Unknown tool: {tool_name}. Available tools: {list(self.tools.keys())}"

        for param in defn.parameters:
            if param.required and param.name not in kwargs:
                return (f"Missing required parameter '{param.name}' for tool '{tool_name}'. "
                        f"Expected type: {param.type}. Description: {param.description}")
            if param.name in kwargs:
                value = kwargs[param.name]
                expected = self.TYPE_MAP.get(param.type)
                if expected and not isinstance(value, expected):
                    return (f"Parameter '{param.name}' has wrong type: got {type(value).__name__}, "
                            f"expected {param.type}. Value was: {value!r}")
                if param.enum and value not in param.enum:
                    return (f"Parameter '{param.name}' value '{value}' not in allowed values: "
                            f"{param.enum}")
                if param.min_value is not None and isinstance(value, (int, float)):
                    if value < param.min_value:
                        return f"Parameter '{param.name}' value {value} is below minimum {param.min_value}"
                if param.max_value is not None and isinstance(value, (int, float)):
                    if value > param.max_value:
                        return f"Parameter '{param.name}' value {value} exceeds maximum {param.max_value}"
        return None

    def call(self, tool_name: str, **kwargs) -> ToolResult:
        """Call a tool with validation and error handling."""
        start = time.time()
        # Validate
        error = self.validate_input(tool_name, kwargs)
        if error:
            result = ToolResult(False, error=error, tool_name=tool_name)
            self.call_history.append({"tool": tool_name, "args": kwargs, "result": result.to_dict()})
            return result
        # Execute
        try:
            output = self.tools[tool_name](**kwargs)
            elapsed = time.time() - start
            result = ToolResult(True, result=output, tool_name=tool_name, execution_time=elapsed)
        except Exception as e:
            elapsed = time.time() - start
            result = ToolResult(False, error=f"{type(e).__name__}: {e}",
                                tool_name=tool_name, execution_time=elapsed)
        self.call_history.append({"tool": tool_name, "args": kwargs, "result": result.to_dict()})
        return result

    def discover(self, category: str = None) -> str:
        """Return a formatted list of available tools for the LLM."""
        lines = ["Available tools:\n"]
        for name, defn in sorted(self.definitions.items()):
            if category and defn.category != category:
                continue
            lines.append(defn.to_prompt_string())
            lines.append("")
        return "\n".join(lines)

    def get_stats(self) -> dict:
        total = len(self.call_history)
        successes = sum(1 for c in self.call_history if c["result"]["success"])
        return {"total_calls": total, "successes": successes, "failures": total - successes,
                "success_rate": f"{successes/total:.1%}" if total else "N/A"}

registry = ToolRegistry()
print("✓ ToolRegistry initialized")
print(f"  Type validation: {list(ToolRegistry.TYPE_MAP.keys())}")

## 4. Complex Tool Types

Real-world tools are more than stateless functions. Let's build three advanced patterns.

### 4.1 — Stateful Tools

Some tools maintain state across calls: database connections, session data, running totals. We wrap these in a class that the registry can manage.

In [ ]:
class StatefulKeyValueStore:
    """A stateful tool — an in-memory key-value store that persists across calls."""

    def __init__(self):
        self.store: Dict[str, Any] = {}
        self.access_log: List[dict] = []

    def set(self, key: str, value: str) -> dict:
        """Set a key-value pair."""
        old = self.store.get(key)
        self.store[key] = value
        self.access_log.append({"op": "set", "key": key, "time": time.time()})
        return {"status": "updated" if old else "created", "key": key, "value": value}

    def get(self, key: str) -> dict:
        """Get a value by key."""
        self.access_log.append({"op": "get", "key": key, "time": time.time()})
        if key in self.store:
            return {"found": True, "key": key, "value": self.store[key]}
        return {"found": False, "key": key, "error": f"Key '{key}' not found. Available keys: {list(self.store.keys())}"}

    def delete(self, key: str) -> dict:
        """Delete a key."""
        if key in self.store:
            del self.store[key]
            return {"deleted": True, "key": key}
        return {"deleted": False, "error": f"Key '{key}' not found"}

    def list_keys(self) -> dict:
        return {"keys": list(self.store.keys()), "count": len(self.store)}

# Create and register
kv = StatefulKeyValueStore()

kv_set_def = ToolDefinition(
    name="kv_set", description="Store a key-value pair in the persistent store.",
    parameters=[
        ParameterSchema("key", "string", "The key to store"),
        ParameterSchema("value", "string", "The value to store"),
    ],
    return_type="dict", return_description="Status dict with 'status', 'key', 'value'",
    examples=[{"input": {"key": "user_name", "value": "Alice"}, "output": {"status": "created", "key": "user_name", "value": "Alice"}}],
    category="storage",
)
kv_get_def = ToolDefinition(
    name="kv_get", description="Retrieve a value by key from the persistent store.",
    parameters=[ParameterSchema("key", "string", "The key to look up")],
    return_type="dict", return_description="Dict with 'found', 'key', 'value' or 'error'",
    category="storage",
)

registry.register(kv_set_def, kv.set)
registry.register(kv_get_def, kv.get)

# Test stateful tool
print("\n--- Stateful Tool Demo ---")
print(registry.call("kv_set", key="capital_france", value="Paris"))
print(registry.call("kv_set", key="capital_japan", value="Tokyo"))
print(registry.call("kv_get", key="capital_france"))
print(registry.call("kv_get", key="unknown_key"))

### 4.2 — Confirmation Tools

Some operations are destructive or irreversible. A confirmation tool adds a "dry run" mode so the agent can preview the effect before committing.

In [ ]:
class ConfirmationTool:
    """Tool that requires explicit confirmation for destructive operations."""

    def __init__(self):
        self.pending_actions: Dict[str, dict] = {}
        self.action_counter = 0

    def request_delete(self, resource_type: str, resource_id: str) -> dict:
        """Request deletion — returns a confirmation token instead of deleting immediately."""
        self.action_counter += 1
        token = f"confirm_{self.action_counter}"
        self.pending_actions[token] = {
            "action": "delete",
            "resource_type": resource_type,
            "resource_id": resource_id,
            "created": time.time(),
        }
        return {
            "status": "confirmation_required",
            "message": f"To delete {resource_type} '{resource_id}', call confirm_action with token '{token}'",
            "token": token,
            "preview": f"This will permanently delete {resource_type} '{resource_id}'",
        }

    def confirm_action(self, token: str) -> dict:
        """Confirm a pending action using its token."""
        if token not in self.pending_actions:
            return {"success": False, "error": f"Invalid or expired token: {token}. Pending tokens: {list(self.pending_actions.keys())}"}
        action = self.pending_actions.pop(token)
        # In production, this would actually perform the deletion
        return {
            "success": True,
            "action": action["action"],
            "resource_type": action["resource_type"],
            "resource_id": action["resource_id"],
            "message": f"Successfully deleted {action['resource_type']} '{action['resource_id']}'",
        }

confirm_tool = ConfirmationTool()

confirm_delete_def = ToolDefinition(
    name="request_delete", description="Request deletion of a resource. Returns a confirmation token — the deletion is NOT performed until you call confirm_action with the token.",
    parameters=[
        ParameterSchema("resource_type", "string", "Type of resource (e.g. 'file', 'record', 'user')"),
        ParameterSchema("resource_id", "string", "ID of the resource to delete"),
    ],
    return_type="dict", return_description="Confirmation token and preview of the action",
    category="admin",
)
confirm_action_def = ToolDefinition(
    name="confirm_action", description="Confirm a previously requested destructive action using the token returned by request_delete.",
    parameters=[ParameterSchema("token", "string", "The confirmation token from request_delete")],
    return_type="dict", return_description="Result of the confirmed action",
    category="admin",
)

registry.register(confirm_delete_def, confirm_tool.request_delete)
registry.register(confirm_action_def, confirm_tool.confirm_action)

# Demo confirmation flow
print("--- Confirmation Flow Demo ---")
result1 = registry.call("request_delete", resource_type="file", resource_id="important_data.csv")
print(f"Step 1 - Request: {result1}")
token = result1.result["token"]
result2 = registry.call("confirm_action", token=token)
print(f"Step 2 - Confirm: {result2}")
print(f"\nInvalid token test:")
print(registry.call("confirm_action", token="bogus_token"))

### 4.3 — Composed Tools

A composed tool calls other tools as sub-steps. For example, a "backup and delete" tool calls the backup tool first, then the delete tool. This keeps individual tools simple while enabling complex workflows.

In [ ]:
def composed_store_and_verify(key: str, value: str) -> dict:
    """Composed tool: store a value, then immediately read it back to verify."""
    # Step 1: store
    set_result = registry.call("kv_set", key=key, value=value)
    if not set_result.success:
        return {"success": False, "step": "store", "error": set_result.error}
    # Step 2: verify by reading back
    get_result = registry.call("kv_get", key=key)
    if not get_result.success:
        return {"success": False, "step": "verify", "error": get_result.error}
    # Step 3: compare
    stored_value = get_result.result.get("value")
    verified = stored_value == value
    return {
        "success": verified,
        "key": key,
        "stored_value": stored_value,
        "verified": verified,
        "steps_taken": ["kv_set", "kv_get", "compare"],
    }

composed_def = ToolDefinition(
    name="store_and_verify",
    description="Store a key-value pair and immediately verify it was stored correctly. Uses kv_set and kv_get internally.",
    parameters=[
        ParameterSchema("key", "string", "The key to store"),
        ParameterSchema("value", "string", "The value to store"),
    ],
    return_type="dict", return_description="Dict with 'success', 'verified', 'steps_taken'",
    examples=[{"input": {"key": "test", "value": "hello"}, "output": {"success": True, "verified": True}}],
    category="storage",
)

registry.register(composed_def, composed_store_and_verify)

# Test composed tool
print("--- Composed Tool Demo ---")
result = registry.call("store_and_verify", key="pi_approx", value="3.14159")
print(json.dumps(result.to_dict(), indent=2))

## 5. Input Validation — Making Errors LLM-Friendly

When validation fails, the error message is the most important part: it must tell the LLM **exactly** what went wrong and **how to fix it**. Vague errors like "invalid input" cause the agent to spiral.

### Key Principle: Error Messages as Corrective Prompts

Each error message should contain:
1. **What went wrong** — `"Parameter 'count' has wrong type"`
2. **What was expected** — `"expected integer, got string"`
3. **How to fix** — `"Pass a number like 5, not a string like '5'"`

In [ ]:
def validate_with_helpful_errors(definition: ToolDefinition, kwargs: dict) -> Optional[str]:
    """Enhanced validation with extremely helpful error messages for LLM recovery."""
    errors = []

    # Check for unknown parameters
    known_names = {p.name for p in definition.parameters}
    unknown = set(kwargs.keys()) - known_names
    if unknown:
        errors.append(
            f"Unknown parameter(s): {unknown}. "
            f"Valid parameters for '{definition.name}': {sorted(known_names)}"
        )

    for param in definition.parameters:
        # Missing required parameter
        if param.required and param.name not in kwargs:
            example = ""
            if definition.examples:
                ex_input = definition.examples[0].get("input", {})
                if param.name in ex_input:
                    example = f" Example: {param.name}={ex_input[param.name]!r}"
            errors.append(
                f"Missing required parameter '{param.name}' ({param.type}): "
                f"{param.description}.{example}"
            )
            continue

        if param.name not in kwargs:
            continue

        value = kwargs[param.name]

        # Type check
        type_map = {"string": str, "number": (int, float), "integer": int,
                     "boolean": bool, "list": list, "dict": dict}
        expected = type_map.get(param.type)
        if expected and not isinstance(value, expected):
            fix_hint = ""
            if param.type == "integer" and isinstance(value, str):
                fix_hint = f" Try passing {param.name}={value} without quotes."
            elif param.type == "string" and isinstance(value, (int, float)):
                fix_hint = f' Try passing {param.name}="{value}".'
            errors.append(
                f"Parameter '{param.name}': expected {param.type}, got "
                f"{type(value).__name__} ({value!r}).{fix_hint}"
            )

        # Enum check
        if param.enum and value not in param.enum:
            errors.append(
                f"Parameter '{param.name}': value '{value}' is not allowed. "
                f"Choose one of: {param.enum}"
            )

        # Range check
        if isinstance(value, (int, float)):
            if param.min_value is not None and value < param.min_value:
                errors.append(f"Parameter '{param.name}': {value} is below minimum {param.min_value}")
            if param.max_value is not None and value > param.max_value:
                errors.append(f"Parameter '{param.name}': {value} exceeds maximum {param.max_value}")

    if errors:
        return "TOOL CALL ERROR for '" + definition.name + "':\n" + "\n".join(f"  • {e}" for e in errors)
    return None

# Demonstrate with intentional mistakes
test_def = ToolDefinition(
    name="example_tool", description="An example tool",
    parameters=[
        ParameterSchema("count", "integer", "Number of items", required=True, min_value=1, max_value=100),
        ParameterSchema("mode", "string", "Operating mode", enum=["fast", "slow", "balanced"]),
    ],
    return_type="string", return_description="Result string",
)

print("=== Validation Error Messages ===\n")

# Missing required param
err = validate_with_helpful_errors(test_def, {"mode": "fast"})
print(f"Test 1 (missing required):\n{err}\n")

# Wrong type
err = validate_with_helpful_errors(test_def, {"count": "five", "mode": "fast"})
print(f"Test 2 (wrong type):\n{err}\n")

# Out of range
err = validate_with_helpful_errors(test_def, {"count": 999, "mode": "fast"})
print(f"Test 3 (out of range):\n{err}\n")

# Invalid enum
err = validate_with_helpful_errors(test_def, {"count": 5, "mode": "turbo"})
print(f"Test 4 (invalid enum):\n{err}\n")

# Valid input
err = validate_with_helpful_errors(test_def, {"count": 5, "mode": "fast"})
print(f"Test 5 (valid): {err}")

## 6. Tool Testing Harness

You wouldn't ship code without tests. Tools need testing too — especially since an LLM will call them in unpredictable ways.

We build a test harness that:
1. Runs **happy-path** tests (expected inputs → expected outputs)
2. Runs **edge-case** tests (boundary values, empty inputs)
3. Runs **error-case** tests (intentionally bad inputs → graceful errors)
4. Reports results in a clear table

In [ ]:
@dataclass
class ToolTestCase:
    """A single test case for a tool."""
    name: str
    tool_name: str
    inputs: dict
    expected_success: bool
    expected_contains: Optional[str] = None   # substring to find in result
    expected_error_contains: Optional[str] = None

class ToolTestHarness:
    """Automated testing framework for tools."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.results: List[dict] = []

    def run_test(self, test: ToolTestCase) -> dict:
        result = self.registry.call(test.tool_name, **test.inputs)
        passed = True
        reason = "OK"

        if result.success != test.expected_success:
            passed = False
            reason = f"Expected success={test.expected_success}, got {result.success}"
        elif test.expected_contains and test.expected_success:
            result_str = str(result.result)
            if test.expected_contains not in result_str:
                passed = False
                reason = f"Result missing '{test.expected_contains}'"
        elif test.expected_error_contains and not test.expected_success:
            if test.expected_error_contains not in str(result.error):
                passed = False
                reason = f"Error missing '{test.expected_error_contains}'"

        entry = {"test": test.name, "tool": test.tool_name, "passed": passed, "reason": reason}
        self.results.append(entry)
        return entry

    def run_suite(self, tests: List[ToolTestCase]) -> None:
        print(f"Running {len(tests)} tests...\n")
        print(f"{'Test':<35} {'Tool':<20} {'Result':<8} {'Reason'}")
        print("-" * 90)
        for test in tests:
            entry = self.run_test(test)
            status = "✓ PASS" if entry["passed"] else "✗ FAIL"
            print(f"{entry['test']:<35} {entry['tool']:<20} {status:<8} {entry['reason']}")
        passed = sum(1 for r in self.results if r["passed"])
        print(f"\n{'='*90}")
        print(f"Results: {passed}/{len(self.results)} passed ({passed/len(self.results):.0%})")

# Define test cases for KV store
kv_tests = [
    ToolTestCase("set_basic", "kv_set", {"key": "color", "value": "blue"}, True, "created"),
    ToolTestCase("set_overwrite", "kv_set", {"key": "color", "value": "red"}, True, "updated"),
    ToolTestCase("get_existing", "kv_get", {"key": "color"}, True, "red"),
    ToolTestCase("get_missing", "kv_get", {"key": "nonexistent"}, True, "not found"),
    ToolTestCase("set_empty_key", "kv_set", {"key": "", "value": "test"}, True),
    ToolTestCase("verify_composed", "store_and_verify", {"key": "tested", "value": "yes"}, True, "verified"),
]

harness = ToolTestHarness(registry)
harness.run_suite(kv_tests)

## 7. Building 10 Production-Quality Tools

Now we build a comprehensive toolkit. Each tool follows our design principles: clear schema, input validation, helpful errors, usage examples.

In [ ]:
# ---- Tool 1: Calculator ----
import operator
import math as math_module

SAFE_OPS = {
    'sqrt': math_module.sqrt, 'abs': abs, 'round': round,
    'sin': math_module.sin, 'cos': math_module.cos, 'tan': math_module.tan,
    'log': math_module.log, 'log10': math_module.log10,
    'pi': math_module.pi, 'e': math_module.e,
    'pow': pow, 'min': min, 'max': max,
}

def tool_calculator(expression: str) -> dict:
    """Safely evaluate a mathematical expression."""
    # Remove dangerous characters
    cleaned = expression.strip()
    # Only allow digits, operators, parens, dots, spaces, and known function names
    allowed_chars = set("0123456789+-*/().,%^ ")
    for fname in SAFE_OPS:
        cleaned = cleaned.replace(fname, "")
    if not all(c in allowed_chars for c in cleaned):
        return {"success": False, "error": f"Expression contains disallowed characters. Only math operations are permitted."}
    try:
        result = eval(expression, {"__builtins__": {}}, SAFE_OPS)
        return {"success": True, "result": result, "expression": expression}
    except Exception as e:
        return {"success": False, "error": f"Math error: {e}. Check your expression syntax."}

calc_def = ToolDefinition(
    name="calculator", description="Safely evaluate math expressions. Supports +,-,*,/,**,sqrt,sin,cos,tan,log,pi,e,abs,round,pow,min,max.",
    parameters=[ParameterSchema("expression", "string", "Math expression like '2+3*4' or 'sqrt(144)'")],
    return_type="dict", return_description="{'success': bool, 'result': number} or {'success': False, 'error': str}",
    examples=[
        {"input": {"expression": "2 + 3 * 4"}, "output": {"success": True, "result": 14}},
        {"input": {"expression": "sqrt(144) + pi"}, "output": {"success": True, "result": 15.14159}},
    ],
    category="math", version="2.0",
)
registry.register(calc_def, tool_calculator)

# ---- Tool 2: String Utilities ----
def tool_string_utils(text: str, operation: str) -> dict:
    """Perform string operations."""
    ops = {
        "upper": lambda t: t.upper(),
        "lower": lambda t: t.lower(),
        "title": lambda t: t.title(),
        "reverse": lambda t: t[::-1],
        "length": lambda t: len(t),
        "word_count": lambda t: len(t.split()),
        "strip": lambda t: t.strip(),
        "char_count": lambda t: dict(sorted(((c, t.count(c)) for c in set(t) if c.strip()), key=lambda x: -x[1])[:10]),
    }
    if operation not in ops:
        return {"success": False, "error": f"Unknown operation '{operation}'. Available: {list(ops.keys())}"}
    try:
        result = ops[operation](text)
        return {"success": True, "operation": operation, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e)}

string_def = ToolDefinition(
    name="string_utils", description="Perform text operations: upper, lower, title, reverse, length, word_count, strip, char_count.",
    parameters=[
        ParameterSchema("text", "string", "Input text"),
        ParameterSchema("operation", "string", "Operation to perform",
                        enum=["upper", "lower", "title", "reverse", "length", "word_count", "strip", "char_count"]),
    ],
    return_type="dict", return_description="{'success': bool, 'operation': str, 'result': ...}",
    category="text",
)
registry.register(string_def, tool_string_utils)

# Quick test
print("\n--- Quick tests ---")
print(registry.call("calculator", expression="sqrt(144) + 10"))
print(registry.call("string_utils", text="Hello World", operation="word_count"))

In [ ]:
# ---- Tool 3: List Operations ----
def tool_list_ops(items: list, operation: str) -> dict:
    """Perform operations on lists."""
    ops = {
        "sort": lambda lst: sorted(lst),
        "reverse": lambda lst: list(reversed(lst)),
        "unique": lambda lst: list(dict.fromkeys(lst)),
        "length": lambda lst: len(lst),
        "sum": lambda lst: sum(lst) if all(isinstance(x, (int, float)) for x in lst) else "Error: not all items are numbers",
        "min": lambda lst: min(lst) if lst else "Error: empty list",
        "max": lambda lst: max(lst) if lst else "Error: empty list",
        "flatten": lambda lst: [item for sublist in lst for item in (sublist if isinstance(sublist, list) else [sublist])],
        "frequencies": lambda lst: dict(sorted(((x, lst.count(x)) for x in set(map(str, lst))), key=lambda p: -p[1])),
    }
    if operation not in ops:
        return {"success": False, "error": f"Unknown operation '{operation}'. Available: {list(ops.keys())}"}
    try:
        result = ops[operation](items)
        if isinstance(result, str) and result.startswith("Error"):
            return {"success": False, "error": result}
        return {"success": True, "operation": operation, "result": result, "input_length": len(items)}
    except Exception as e:
        return {"success": False, "error": f"{type(e).__name__}: {e}"}

list_def = ToolDefinition(
    name="list_ops", description="List operations: sort, reverse, unique, length, sum, min, max, flatten, frequencies.",
    parameters=[
        ParameterSchema("items", "list", "The list to operate on"),
        ParameterSchema("operation", "string", "Operation name",
                        enum=["sort", "reverse", "unique", "length", "sum", "min", "max", "flatten", "frequencies"]),
    ],
    return_type="dict", return_description="{'success': bool, 'result': ...}",
    category="data",
)
registry.register(list_def, tool_list_ops)

# ---- Tool 4: Dict Operations ----
def tool_dict_ops(data: dict, operation: str, key: str = None) -> dict:
    """Perform operations on dictionaries."""
    ops_no_key = {
        "keys": lambda d, _: list(d.keys()),
        "values": lambda d, _: list(d.values()),
        "length": lambda d, _: len(d),
        "sorted_keys": lambda d, _: dict(sorted(d.items())),
        "invert": lambda d, _: {str(v): k for k, v in d.items()},
    }
    ops_with_key = {
        "get": lambda d, k: d.get(k, f"Key '{k}' not found. Available: {list(d.keys())}"),
        "has_key": lambda d, k: k in d,
        "delete": lambda d, k: {dk: dv for dk, dv in d.items() if dk != k},
    }
    all_ops = {**ops_no_key, **ops_with_key}
    if operation not in all_ops:
        return {"success": False, "error": f"Unknown operation '{operation}'. Available: {list(all_ops.keys())}"}
    if operation in ops_with_key and key is None:
        return {"success": False, "error": f"Operation '{operation}' requires a 'key' parameter."}
    try:
        result = all_ops[operation](data, key)
        return {"success": True, "operation": operation, "result": result}
    except Exception as e:
        return {"success": False, "error": str(e)}

dict_def = ToolDefinition(
    name="dict_ops", description="Dictionary operations: keys, values, length, sorted_keys, invert, get, has_key, delete.",
    parameters=[
        ParameterSchema("data", "dict", "The dictionary to operate on"),
        ParameterSchema("operation", "string", "Operation name",
                        enum=["keys", "values", "length", "sorted_keys", "invert", "get", "has_key", "delete"]),
        ParameterSchema("key", "string", "Key for get/has_key/delete operations", required=False),
    ],
    return_type="dict", return_description="{'success': bool, 'result': ...}",
    category="data",
)
registry.register(dict_def, tool_dict_ops)

print("\n--- Quick tests ---")
print(registry.call("list_ops", items=[3, 1, 4, 1, 5, 9], operation="sort"))
print(registry.call("dict_ops", data={"a": 1, "b": 2, "c": 3}, operation="keys"))

In [ ]:
# ---- Tool 5: Date & Time ----
from datetime import datetime, timedelta

def tool_date_time(operation: str, date_str: str = None, days: int = None, format: str = None) -> dict:
    """Date and time operations."""
    try:
        if operation == "now":
            now = datetime.now()
            return {"success": True, "result": now.strftime("%Y-%m-%d %H:%M:%S"), "timestamp": now.timestamp()}
        elif operation == "parse":
            if not date_str:
                return {"success": False, "error": "Operation 'parse' requires 'date_str'. Example: date_str='2024-01-15'"}
            fmt = format or "%Y-%m-%d"
            dt = datetime.strptime(date_str, fmt)
            return {"success": True, "parsed": dt.strftime("%Y-%m-%d %H:%M:%S"),
                    "weekday": dt.strftime("%A"), "day_of_year": dt.timetuple().tm_yday}
        elif operation == "add_days":
            if not date_str:
                return {"success": False, "error": "Operation 'add_days' requires 'date_str'."}
            if days is None:
                return {"success": False, "error": "Operation 'add_days' requires 'days' parameter."}
            dt = datetime.strptime(date_str, format or "%Y-%m-%d")
            result_dt = dt + timedelta(days=days)
            return {"success": True, "original": date_str, "days_added": days,
                    "result": result_dt.strftime("%Y-%m-%d"), "weekday": result_dt.strftime("%A")}
        elif operation == "diff":
            if not date_str or not format:
                return {"success": False, "error": "Operation 'diff' requires 'date_str' (first date) and 'format' (second date in YYYY-MM-DD)."}
            dt1 = datetime.strptime(date_str, "%Y-%m-%d")
            dt2 = datetime.strptime(format, "%Y-%m-%d")
            diff = abs((dt2 - dt1).days)
            return {"success": True, "date1": date_str, "date2": format, "difference_days": diff}
        else:
            return {"success": False, "error": f"Unknown operation '{operation}'. Available: now, parse, add_days, diff"}
    except ValueError as e:
        return {"success": False, "error": f"Date parsing error: {e}. Use format YYYY-MM-DD."}

date_def = ToolDefinition(
    name="date_time", description="Date/time operations: now, parse, add_days, diff.",
    parameters=[
        ParameterSchema("operation", "string", "Operation: now, parse, add_days, diff",
                        enum=["now", "parse", "add_days", "diff"]),
        ParameterSchema("date_str", "string", "Date string (YYYY-MM-DD)", required=False),
        ParameterSchema("days", "integer", "Number of days to add", required=False),
        ParameterSchema("format", "string", "Date format string or second date for diff", required=False),
    ],
    return_type="dict", return_description="Operation result dict",
    category="utility",
)
registry.register(date_def, tool_date_time)

# ---- Tool 6: Text Statistics ----
import statistics

def tool_text_stats(text: str) -> dict:
    """Compute comprehensive text statistics."""
    words = text.split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    word_lengths = [len(w) for w in words]
    return {
        "success": True,
        "characters": len(text),
        "characters_no_spaces": len(text.replace(" ", "")),
        "words": len(words),
        "sentences": len(sentences),
        "paragraphs": text.count("\n\n") + 1,
        "avg_word_length": round(statistics.mean(word_lengths), 1) if word_lengths else 0,
        "avg_sentence_length": round(len(words) / max(len(sentences), 1), 1),
        "longest_word": max(words, key=len) if words else "",
        "unique_words": len(set(w.lower().strip(".,!?;:") for w in words)),
        "lexical_diversity": round(len(set(w.lower().strip(".,!?;:") for w in words)) / max(len(words), 1), 3),
    }

text_stats_def = ToolDefinition(
    name="text_stats", description="Compute comprehensive text statistics: word count, sentence count, avg lengths, lexical diversity.",
    parameters=[ParameterSchema("text", "string", "Text to analyze")],
    return_type="dict", return_description="Statistics dictionary with counts, averages, and diversity metrics",
    category="text",
)
registry.register(text_stats_def, tool_text_stats)

print("--- Quick tests ---")
print(registry.call("date_time", operation="now"))
stats_result = registry.call("text_stats", text="The quick brown fox jumps over the lazy dog. The dog barked loudly.")
print(json.dumps(stats_result.to_dict(), indent=2))

In [ ]:
# ---- Tool 7: Format Converter ----
import csv
import io

def tool_format_converter(data: str, from_format: str, to_format: str) -> dict:
    """Convert data between formats."""
    try:
        # Parse input
        if from_format == "json":
            parsed = json.loads(data)
        elif from_format == "csv":
            reader = csv.DictReader(io.StringIO(data))
            parsed = list(reader)
        elif from_format == "text_lines":
            parsed = [line.strip() for line in data.strip().split("\n") if line.strip()]
        else:
            return {"success": False, "error": f"Unknown from_format '{from_format}'. Use: json, csv, text_lines"}

        # Convert to output format
        if to_format == "json":
            result = json.dumps(parsed, indent=2)
        elif to_format == "csv":
            if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
                output = io.StringIO()
                writer = csv.DictWriter(output, fieldnames=parsed[0].keys())
                writer.writeheader()
                writer.writerows(parsed)
                result = output.getvalue()
            else:
                return {"success": False, "error": "CSV output requires list of dicts"}
        elif to_format == "markdown_table":
            if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
                headers = list(parsed[0].keys())
                lines = ["| " + " | ".join(headers) + " |",
                          "| " + " | ".join(["---"] * len(headers)) + " |"]
                for row in parsed:
                    lines.append("| " + " | ".join(str(row.get(h, "")) for h in headers) + " |")
                result = "\n".join(lines)
            else:
                return {"success": False, "error": "Markdown table requires list of dicts"}
        else:
            return {"success": False, "error": f"Unknown to_format '{to_format}'. Use: json, csv, markdown_table"}

        return {"success": True, "result": result, "from": from_format, "to": to_format}
    except Exception as e:
        return {"success": False, "error": f"Conversion error: {e}"}

format_def = ToolDefinition(
    name="format_converter", description="Convert data between formats: json, csv, text_lines, markdown_table.",
    parameters=[
        ParameterSchema("data", "string", "Input data as string"),
        ParameterSchema("from_format", "string", "Source format", enum=["json", "csv", "text_lines"]),
        ParameterSchema("to_format", "string", "Target format", enum=["json", "csv", "markdown_table"]),
    ],
    return_type="dict", return_description="{'success': bool, 'result': converted_string}",
    category="utility",
)
registry.register(format_def, tool_format_converter)

# ---- Tool 8: Data Validator ----
def tool_data_validator(data: dict, rules: dict) -> dict:
    """Validate a data dictionary against rules."""
    errors = []
    warnings = []
    for field_name, field_rules in rules.items():
        value = data.get(field_name)
        # Required check
        if field_rules.get("required", False) and (value is None or value == ""):
            errors.append(f"Field '{field_name}' is required but missing or empty")
            continue
        if value is None:
            continue
        # Type check
        expected_type = field_rules.get("type")
        if expected_type == "string" and not isinstance(value, str):
            errors.append(f"Field '{field_name}': expected string, got {type(value).__name__}")
        elif expected_type == "number" and not isinstance(value, (int, float)):
            errors.append(f"Field '{field_name}': expected number, got {type(value).__name__}")
        # Min/max
        if "min" in field_rules and isinstance(value, (int, float)) and value < field_rules["min"]:
            errors.append(f"Field '{field_name}': {value} < minimum {field_rules['min']}")
        if "max" in field_rules and isinstance(value, (int, float)) and value > field_rules["max"]:
            errors.append(f"Field '{field_name}': {value} > maximum {field_rules['max']}")
        # Pattern
        if "pattern" in field_rules and isinstance(value, str):
            if not re.match(field_rules["pattern"], value):
                errors.append(f"Field '{field_name}': '{value}' doesn't match pattern '{field_rules['pattern']}'")
        # Allowed values
        if "allowed" in field_rules and value not in field_rules["allowed"]:
            errors.append(f"Field '{field_name}': '{value}' not in {field_rules['allowed']}")
        # Min length
        if "min_length" in field_rules and isinstance(value, str) and len(value) < field_rules["min_length"]:
            warnings.append(f"Field '{field_name}': length {len(value)} < recommended minimum {field_rules['min_length']}")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "fields_checked": len(rules),
    }

validator_def = ToolDefinition(
    name="data_validator", description="Validate a data dict against rules (required, type, min/max, pattern, allowed values).",
    parameters=[
        ParameterSchema("data", "dict", "Data dictionary to validate"),
        ParameterSchema("rules", "dict", "Validation rules per field"),
    ],
    return_type="dict", return_description="{'valid': bool, 'errors': [...], 'warnings': [...]}",
    category="utility",
)
registry.register(validator_def, tool_data_validator)

# Tests
test_json = '[{"name":"Alice","age":30},{"name":"Bob","age":25}]'
print(registry.call("format_converter", data=test_json, from_format="json", to_format="markdown_table"))
print()
print(registry.call("data_validator",
    data={"name": "Alice", "age": 150, "email": "bad-email"},
    rules={
        "name": {"required": True, "type": "string", "min_length": 2},
        "age": {"required": True, "type": "number", "min": 0, "max": 130},
        "email": {"required": True, "type": "string", "pattern": r"^[\w.]+@[\w.]+$"},
    }))

In [ ]:
# ---- Tool 9: Advanced Math ----
from functools import reduce

def tool_math_advanced(operation: str, numbers: list) -> dict:
    """Advanced math operations on number lists."""
    if not numbers:
        return {"success": False, "error": "Empty number list. Provide at least one number."}
    if not all(isinstance(n, (int, float)) for n in numbers):
        return {"success": False, "error": f"All items must be numbers. Got types: {[type(n).__name__ for n in numbers]}"}
    try:
        if operation == "mean":
            result = statistics.mean(numbers)
        elif operation == "median":
            result = statistics.median(numbers)
        elif operation == "stdev":
            if len(numbers) < 2:
                return {"success": False, "error": "Standard deviation requires at least 2 numbers"}
            result = statistics.stdev(numbers)
        elif operation == "variance":
            if len(numbers) < 2:
                return {"success": False, "error": "Variance requires at least 2 numbers"}
            result = statistics.variance(numbers)
        elif operation == "product":
            result = reduce(lambda a, b: a * b, numbers)
        elif operation == "gcd":
            from math import gcd
            int_nums = [int(n) for n in numbers]
            result = reduce(gcd, int_nums)
        elif operation == "lcm":
            from math import gcd
            def lcm(a, b): return abs(a * b) // gcd(a, b)
            int_nums = [int(n) for n in numbers]
            result = reduce(lcm, int_nums)
        elif operation == "percentile_25":
            sorted_nums = sorted(numbers)
            idx = len(sorted_nums) * 0.25
            result = sorted_nums[int(idx)]
        elif operation == "percentile_75":
            sorted_nums = sorted(numbers)
            idx = len(sorted_nums) * 0.75
            result = sorted_nums[min(int(idx), len(sorted_nums) - 1)]
        elif operation == "range":
            result = max(numbers) - min(numbers)
        else:
            return {"success": False, "error": f"Unknown operation '{operation}'. Available: mean, median, stdev, variance, product, gcd, lcm, percentile_25, percentile_75, range"}
        return {"success": True, "operation": operation, "result": round(result, 6), "count": len(numbers)}
    except Exception as e:
        return {"success": False, "error": f"{type(e).__name__}: {e}"}

math_adv_def = ToolDefinition(
    name="math_advanced", description="Statistical and advanced math: mean, median, stdev, variance, product, gcd, lcm, percentile_25, percentile_75, range.",
    parameters=[
        ParameterSchema("operation", "string", "Operation name",
                        enum=["mean", "median", "stdev", "variance", "product", "gcd", "lcm", "percentile_25", "percentile_75", "range"]),
        ParameterSchema("numbers", "list", "List of numbers"),
    ],
    return_type="dict", return_description="{'success': bool, 'result': number}",
    category="math",
)
registry.register(math_adv_def, tool_math_advanced)

# ---- Tool 10: Encoding Tools ----
import base64
import hashlib
import urllib.parse

def tool_encoding(text: str, operation: str) -> dict:
    """Text encoding/decoding operations."""
    try:
        if operation == "base64_encode":
            result = base64.b64encode(text.encode()).decode()
        elif operation == "base64_decode":
            result = base64.b64decode(text.encode()).decode()
        elif operation == "url_encode":
            result = urllib.parse.quote(text)
        elif operation == "url_decode":
            result = urllib.parse.unquote(text)
        elif operation == "md5":
            result = hashlib.md5(text.encode()).hexdigest()
        elif operation == "sha256":
            result = hashlib.sha256(text.encode()).hexdigest()
        elif operation == "char_codes":
            result = [ord(c) for c in text[:50]]  # limit to 50 chars
        elif operation == "from_char_codes":
            codes = json.loads(text)
            result = "".join(chr(c) for c in codes)
        else:
            return {"success": False, "error": f"Unknown operation '{operation}'. Available: base64_encode, base64_decode, url_encode, url_decode, md5, sha256, char_codes, from_char_codes"}
        return {"success": True, "operation": operation, "result": result}
    except Exception as e:
        return {"success": False, "error": f"{type(e).__name__}: {e}"}

encoding_def = ToolDefinition(
    name="encoding_tools", description="Encoding/decoding: base64_encode/decode, url_encode/decode, md5, sha256, char_codes, from_char_codes.",
    parameters=[
        ParameterSchema("text", "string", "Text to encode/decode"),
        ParameterSchema("operation", "string", "Encoding operation",
                        enum=["base64_encode", "base64_decode", "url_encode", "url_decode", "md5", "sha256", "char_codes", "from_char_codes"]),
    ],
    return_type="dict", return_description="{'success': bool, 'result': str}",
    category="utility",
)
registry.register(encoding_def, tool_encoding)

print("--- Quick tests ---")
print(registry.call("math_advanced", operation="mean", numbers=[10, 20, 30, 40, 50]))
print(registry.call("math_advanced", operation="stdev", numbers=[10, 20, 30, 40, 50]))
print(registry.call("encoding_tools", text="Hello, World!", operation="base64_encode"))
print(registry.call("encoding_tools", text="Hello, World!", operation="sha256"))

### Tool Inventory

Let's survey our complete toolkit and verify everything is registered correctly.

In [ ]:
print("=" * 70)
print(f"{'TOOL REGISTRY SUMMARY':^70}")
print("=" * 70)
print(f"\n{'Tool Name':<25} {'Category':<12} {'Version':<8} {'Params'}")
print("-" * 70)
for name, defn in sorted(registry.definitions.items()):
    param_list = ", ".join(p.name for p in defn.parameters)
    print(f"{name:<25} {defn.category:<12} v{defn.version:<6} {param_list}")

print(f"\nTotal tools registered: {len(registry.tools)}")
categories = defaultdict(list)
for name, defn in registry.definitions.items():
    categories[defn.category].append(name)
print("\nBy category:")
for cat, tools in sorted(categories.items()):
    print(f"  {cat}: {', '.join(tools)}")

## 8. Full Test Suite

Run comprehensive tests across all 10 tools — happy paths, edge cases, and error cases.

In [ ]:
all_tests = [
    # Calculator
    ToolTestCase("calc_basic", "calculator", {"expression": "2 + 3"}, True, "5"),
    ToolTestCase("calc_complex", "calculator", {"expression": "sqrt(16) * 3 + 1"}, True, "13"),
    ToolTestCase("calc_division_by_zero", "calculator", {"expression": "1/0"}, True, "error"),

    # String utils
    ToolTestCase("str_upper", "string_utils", {"text": "hello", "operation": "upper"}, True, "HELLO"),
    ToolTestCase("str_reverse", "string_utils", {"text": "abcdef", "operation": "reverse"}, True, "fedcba"),
    ToolTestCase("str_bad_op", "string_utils", {"text": "test", "operation": "explode"}, True, "Unknown"),

    # List ops
    ToolTestCase("list_sort", "list_ops", {"items": [3, 1, 2], "operation": "sort"}, True),
    ToolTestCase("list_unique", "list_ops", {"items": [1, 2, 2, 3, 3, 3], "operation": "unique"}, True),
    ToolTestCase("list_sum", "list_ops", {"items": [10, 20, 30], "operation": "sum"}, True, "60"),

    # Dict ops
    ToolTestCase("dict_keys", "dict_ops", {"data": {"a": 1}, "operation": "keys"}, True),
    ToolTestCase("dict_get", "dict_ops", {"data": {"x": 42}, "operation": "get", "key": "x"}, True, "42"),

    # Date/time
    ToolTestCase("date_now", "date_time", {"operation": "now"}, True),
    ToolTestCase("date_parse", "date_time", {"operation": "parse", "date_str": "2024-06-15"}, True, "Saturday"),

    # Text stats
    ToolTestCase("stats_basic", "text_stats", {"text": "Hello world. How are you?"}, True),

    # Math advanced
    ToolTestCase("math_mean", "math_advanced", {"operation": "mean", "numbers": [1, 2, 3, 4, 5]}, True, "3"),
    ToolTestCase("math_stdev_one", "math_advanced", {"operation": "stdev", "numbers": [5]}, True, "requires"),

    # Encoding
    ToolTestCase("enc_b64", "encoding_tools", {"text": "test", "operation": "base64_encode"}, True, "dGVzdA"),
    ToolTestCase("enc_md5", "encoding_tools", {"text": "hello", "operation": "md5"}, True),
]

harness2 = ToolTestHarness(registry)
harness2.run_suite(all_tests)

## 9. Agent Integration — Wiring Tools into an Agent Loop

Now we connect all our tools to an LLM agent. The agent receives the tool registry description in its system prompt and can call any tool by emitting a JSON action.

### Architecture

```
User Question
     │
     ▼
┌────────────┐    tool descriptions    ┌──────────────┐
│   Agent    │◄────────────────────────│  ToolRegistry │
│  (LLM)    │    ToolResult            │  (10+ tools)  │
│            │────────────────────────►│               │
└────────────┘    JSON tool call       └──────────────┘
     │
     ▼
Final Answer
```

In [ ]:
class AdvancedToolAgent:
    """Agent that uses the full tool registry with ReAct-style reasoning."""

    def __init__(self, registry: ToolRegistry, max_steps: int = 8):
        self.registry = registry
        self.max_steps = max_steps

    def _build_system_prompt(self) -> str:
        tool_desc = self.registry.discover()
        return f"""You are a helpful assistant with access to tools. Use tools when needed.

{tool_desc}

RESPONSE FORMAT:
When you need a tool, respond with EXACTLY this JSON (no extra text):
{{"thought": "why I need this tool", "tool": "tool_name", "args": {{"param": "value"}}}}

When you have the final answer, respond with:
{{"thought": "reasoning", "answer": "your final answer to the user"}}

Rules:
- Use tools for computation, data manipulation, encoding — don't guess
- If a tool returns an error, read the error message carefully and retry with corrected inputs
- You may chain multiple tool calls to solve complex problems"""

    def run(self, query: str) -> dict:
        messages = [
            {"role": "system", "content": self._build_system_prompt()},
            {"role": "user", "content": query},
        ]
        steps = []
        for step_num in range(self.max_steps):
            response = generate(messages, max_new_tokens=400, temperature=0.3)
            # Try to parse JSON
            try:
                # Extract JSON from response
                json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', response)
                if not json_match:
                    steps.append({"step": step_num, "type": "answer", "content": response})
                    return {"answer": response, "steps": steps, "tool_calls": len(steps) - 1}
                parsed = json.loads(json_match.group())
            except json.JSONDecodeError:
                steps.append({"step": step_num, "type": "answer", "content": response})
                return {"answer": response, "steps": steps, "tool_calls": len(steps) - 1}

            if "answer" in parsed:
                steps.append({"step": step_num, "type": "answer", "content": parsed["answer"]})
                return {"answer": parsed["answer"], "steps": steps, "tool_calls": len([s for s in steps if s["type"] == "tool"])}

            if "tool" in parsed:
                tool_name = parsed["tool"]
                tool_args = parsed.get("args", {})
                thought = parsed.get("thought", "")
                tool_result = self.registry.call(tool_name, **tool_args)
                steps.append({"step": step_num, "type": "tool", "thought": thought,
                              "tool": tool_name, "args": tool_args, "result": tool_result.to_dict()})
                # Feed result back to agent
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Tool result: {json.dumps(tool_result.to_dict())}"})
            else:
                steps.append({"step": step_num, "type": "answer", "content": response})
                return {"answer": response, "steps": steps, "tool_calls": 0}

        return {"answer": "Max steps reached", "steps": steps, "tool_calls": len(steps)}

agent = AdvancedToolAgent(registry)
print("✓ AdvancedToolAgent created")
print(f"  Available tools: {len(registry.tools)}")
print(f"  Max steps per query: {agent.max_steps}")

### 9.1 — Testing the Agent

Let's ask the agent questions that require tool use.

In [ ]:
print("=" * 60)
print("QUERY 1: Math computation")
print("=" * 60)
result = agent.run("What is the square root of 625 plus the base64 encoding of 'hello'?")
print(f"\nFinal answer: {result['answer']}")
print(f"Tool calls made: {result['tool_calls']}")
for step in result["steps"]:
    if step["type"] == "tool":
        print(f"  Step {step['step']}: {step['tool']}({step['args']}) → {step['result']}")

In [ ]:
print("=" * 60)
print("QUERY 2: Data analysis")
print("=" * 60)
result = agent.run("I have these test scores: 85, 92, 78, 95, 88, 91, 76, 84. What's the mean, median, and standard deviation?")
print(f"\nFinal answer: {result['answer']}")
print(f"Tool calls made: {result['tool_calls']}")
for step in result["steps"]:
    if step["type"] == "tool":
        print(f"  Step {step['step']}: {step['tool']}({step['args']}) → {step['result']}")

In [ ]:
print("=" * 60)
print("QUERY 3: Multi-step with error recovery")
print("=" * 60)
result = agent.run("Sort the list [5,2,8,1,9,3], then compute the mean of the sorted list, and tell me the sha256 hash of the result.")
print(f"\nFinal answer: {result['answer']}")
print(f"Tool calls made: {result['tool_calls']}")
for step in result["steps"]:
    if step["type"] == "tool":
        print(f"  Step {step['step']}: {step['tool']}({step['args']})")
        print(f"    → {step['result']}")

## 10. Comparison: Simple Tools vs. Production Tools

Let's quantify the difference. We'll build a "naive" agent with bare tools (no validation, no schemas) and compare error recovery.

In [ ]:
# Simple tools — no validation, no schemas, raw exceptions
def simple_calculator(expression):
    return eval(expression)

def simple_string(text, op):
    return getattr(text, op)()

# Track errors
def test_tool_robustness(tool_func, test_cases, label):
    """Run test cases and track success/failure."""
    results = {"successes": 0, "failures": 0, "errors": []}
    for name, args, kwargs in test_cases:
        try:
            result = tool_func(*args, **kwargs)
            results["successes"] += 1
        except Exception as e:
            results["failures"] += 1
            results["errors"].append(f"{name}: {type(e).__name__}: {e}")
    total = results["successes"] + results["failures"]
    rate = results["successes"] / total if total else 0
    print(f"  {label}: {results['successes']}/{total} passed ({rate:.0%})")
    for err in results["errors"][:3]:
        print(f"    ✗ {err}")
    return rate

print("=" * 60)
print("ROBUSTNESS COMPARISON")
print("=" * 60)

# Test cases that include tricky inputs
calc_cases = [
    ("basic", ("2+3",), {}),
    ("complex", ("sqrt(16)*3",), {}),
    ("divide_zero", ("1/0",), {}),
    ("bad_input", ("hello",), {}),
    ("import_attack", ("__import__('os').listdir()",), {}),
]

print("\nCalculator:")
simple_rate = test_tool_robustness(simple_calculator, calc_cases, "Simple (eval)")

# Production version through registry
prod_calc_results = {"successes": 0, "failures": 0}
for name, args, kwargs in calc_cases:
    result = registry.call("calculator", expression=args[0])
    if result.success or (not result.success and result.error):
        prod_calc_results["successes"] += 1  # Graceful handling = success
total = prod_calc_results["successes"] + prod_calc_results["failures"]
prod_rate = prod_calc_results["successes"] / total if total else 0
print(f"  Production: {prod_calc_results['successes']}/{total} handled gracefully ({prod_rate:.0%})")

print(f"\n{'Metric':<30} {'Simple':<15} {'Production':<15}")
print("-" * 60)
print(f"{'Graceful error handling':<30} {'No — crashes':<15} {'Yes — always':<15}")
print(f"{'Input validation':<30} {'None':<15} {'Full':<15}")
print(f"{'LLM-friendly errors':<30} {'Raw exceptions':<15} {'Guided msgs':<15}")
print(f"{'Schema documentation':<30} {'None':<15} {'Complete':<15}")
print(f"{'Success on tricky inputs':<30} {f'{simple_rate:.0%}':<15} {f'{prod_rate:.0%}':<15}")

### Overall Registry Statistics

Let's look at how our tools performed across all the calls in this notebook.

In [ ]:
stats = registry.get_stats()
print("=" * 50)
print("TOOL REGISTRY LIFETIME STATISTICS")
print("=" * 50)
for k, v in stats.items():
    print(f"  {k}: {v}")

print("\nPer-tool breakdown:")
tool_calls = defaultdict(lambda: {"success": 0, "fail": 0})
for entry in registry.call_history:
    tool = entry["tool"]
    if entry["result"]["success"]:
        tool_calls[tool]["success"] += 1
    else:
        tool_calls[tool]["fail"] += 1

print(f"\n{'Tool':<25} {'Success':<10} {'Fail':<10} {'Total':<10} {'Rate'}")
print("-" * 65)
for tool, counts in sorted(tool_calls.items()):
    total = counts["success"] + counts["fail"]
    rate = counts["success"] / total if total else 0
    print(f"{tool:<25} {counts['success']:<10} {counts['fail']:<10} {total:<10} {rate:.0%}")

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** extend the agent with a new tool. Document what changes and why.

**Exercise 2 — Build:** add error handling for a failure mode discussed in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** trace through the agent loop manually and predict the output before running.

## 11. Summary & Key Takeaways

### What We Built

| Component | Purpose |
|-----------|---------|
| **ToolDefinition** | Formal schema — name, params, types, examples |
| **ToolRegistry** | Central hub — validation, dispatch, error handling |
| **ParameterSchema** | Per-parameter constraints — type, range, enum |
| **ToolResult** | Standardized output — success/error, timing |
| **ToolTestHarness** | Automated testing — happy, edge, error cases |
| **AdvancedToolAgent** | ReAct agent wired to the full registry |

### Design Principles Recap

1. **Single Responsibility** — each tool does one thing. Compose for complex workflows.
2. **Clear Schemas** — the LLM reads your tool descriptions like documentation. Make them excellent.
3. **Predictable Errors** — never crash. Always return structured error info.
4. **Usage Examples** — examples in tool definitions are few-shot prompts for the LLM.
5. **Test Everything** — tools are code. Test them like code.

### Production Checklist

- [ ] Every tool has a ToolDefinition with examples
- [ ] Input validation covers types, ranges, enums, required fields
- [ ] Error messages tell the LLM how to fix the problem
- [ ] Destructive operations use confirmation tokens
- [ ] Stateful tools are explicit about their state
- [ ] Composed tools report which sub-step failed
- [ ] Full test suite with happy, edge, and error cases

### What's Next

In **Notebook 14**, we'll build the most powerful tool of all: a **code execution tool** that lets the agent write and run Python code to solve arbitrary problems. This requires careful sandboxing — the principles from this notebook (validation, error handling, testing) will be essential.

---

*"A tool is only as good as its interface. In human-AI collaboration, the interface IS the tool."*

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the agents/ directory